# **Código de Dinâmica de Máquinas Rotativas**

## **Lista de Exercícios das Aulas** - Marcus Filipe Sousa Reis 12522EMC006

In [1]:
import dmr

import numpy as np
import pandas as pd

import plotly.express as px

np.set_printoptions(formatter={'float': '{:0.3e}'.format})

Dados gerais (material e geometria do rotor)

In [2]:
aco = dmr.Material(
    rho=7800,   # kg/m³
    E=2e11,     # N/m²
)

eixo = dmr.Eixo(
    R=0.01,     # m
    L=0.4,      # m
    material=aco,
)

D1 = dmr.Disco(
    R=0.15,     # m
    r=0.01,     # m
    d=0.03,     # m
    material=aco,
    pos = eixo.L / 3 # m
)

forma = dmr.FuncaoForma(
    L=eixo.L
)

mancal_simetrico = dmr.Mancal(
    kxx=0,
    kyy=0,
    cxx=0,
    cyy=0,
    pos=(2 * eixo.L / 3)
)

### **Rotor Simétrico**

In [3]:
rotor_simetrico = dmr.Rotor(
    eixo=eixo,
    forma=forma,
    mancal=mancal_simetrico,
    disco=D1,
)

In [4]:
rotor_simetrico.plot_campbell(speed_range=(0,9000), title="<b>Diagrama de Campbell</b> F0 = 0", through="equation")

Avaliação da influência da força $F_{0}$ nas frequências naturais

In [5]:
forca_axial = 5000 # Newton

In [6]:
eixo_ftracao = dmr.Eixo(
    R=0.01,             # m
    L=0.4,              # m
    material=aco,
    F0=forca_axial,     # N
)
rotor_tracao = dmr.Rotor(
    eixo=eixo_ftracao,
    forma=forma,
    mancal=mancal_simetrico,
    disco=D1,
)
rotor_tracao.plot_campbell(speed_range=(0,9000), title=f"<b>Diagrama de Campbell</b> F0 = {eixo_ftracao.F0} N", through="equation")

In [7]:
eixo_fcompressao = dmr.Eixo(
    R=0.01,             # m
    L=0.4,              # m
    material=aco,
    F0=-forca_axial,    # N
)
rotor_compressao = dmr.Rotor(
    eixo=eixo_fcompressao,
    forma=forma,
    mancal=mancal_simetrico,
    disco=D1,
)
rotor_compressao.plot_campbell(speed_range=(0, 9000), title=f"<b>Diagrama de Campbell</b> F0 = {eixo_fcompressao.F0} N", through="equation")

#### Resposta ao Desbalanceamento

In [8]:
desbalanceamento = dmr.Desbalanceamento(
    m_u=1e-4,       
    disco=D1
)

In [9]:
rotor_simetrico = dmr.Rotor(
    eixo=eixo,
    forma=forma,
    mancal=mancal_simetrico,
    disco=D1,
    desbalanceamento=desbalanceamento,
)

In [10]:
rotor_simetrico.plot_resposta(speed_range=(0,9000), excitacao="desbalanceamento")

#### Resposta a Força Assíncrona

In [11]:
F = 1.154700
forca_assincrona = dmr.ForcaAssincrona(
    Fx=F,
    Fy=F,
    pos=eixo.L * 2 / 3,
    s=0.5,
)

In [12]:
rotor_simetrico = dmr.Rotor(
    eixo=eixo,
    forma=forma,
    mancal=mancal_simetrico,
    disco=D1,
    desbalanceamento=desbalanceamento,
    forca_assincrona=forca_assincrona
)

In [13]:
rotor_simetrico.plot_resposta(speed_range=(0, 9000), excitacao="assincrona")

#### Resposta a Força fixa no espaço

In [14]:
F = 1.154700
forca_fixa_espaco= dmr.ForcaAssincrona(
    Fx=F,
    Fy=0,
    pos=eixo.L * 2 / 3,
    s=1,
)

In [15]:
rotor_simetrico = dmr.Rotor(
    eixo=eixo,
    forma=forma,
    mancal=mancal_simetrico,
    disco=D1,
    desbalanceamento=desbalanceamento,
    forca_assincrona=forca_fixa_espaco
)

In [16]:
rotor_simetrico.plot_resposta(speed_range=(0,9000), n_points=10000, excitacao="assincrona", w_rpm=4000, speed_unit="Hz")

### **Rotor Assimétrico**

In [17]:
mancal_assimetrico = dmr.Mancal(
    kxx=0,
    kyy=5e5,
    cxx=0,
    cyy=0,
    pos=(2 * eixo.L / 3)
)

In [18]:
rotor_assimetrico = dmr.Rotor(
    eixo=eixo,
    forma=forma,
    mancal=mancal_assimetrico,
    disco=D1,
    desbalanceamento=desbalanceamento,
    forca_assincrona=forca_fixa_espaco,
)

Para este rotor, o **Diagrama de Campbell** é calculado através da excitação pela força fixa no espaço.

In [19]:
rotor_assimetrico.plot_campbell(speed_range=(0,9000), title="<b>Diagrama de Campbell</b>")

#### Resposta ao Desbalanceamento

In [20]:
rotor_assimetrico.plot_resposta(speed_range=(0,9000), n_points=10000, excitacao="desbalanceamento")

### **Rotor Amortecido**

In [21]:
# Coeficientes de Rigidez
kxx = 2e5
kyy = 5e5
# Amortecimento Proporcional
beta = 0.015
mancal_assimetrico_amortecido = dmr.Mancal(
    kxx=kxx,
    kyy=kyy,
    cxx=beta * kxx,
    cyy=beta * kyy,
    pos=(2 * eixo.L / 3)
)

In [22]:
rotor_amortecido = dmr.Rotor(
    eixo=eixo,
    forma=forma,
    mancal=mancal_assimetrico_amortecido,
    disco=D1,
    desbalanceamento=desbalanceamento,
    forca_assincrona=forca_fixa_espaco,
)

In [24]:
rotor_amortecido.plot_campbell(speed_range=(0,9000), title="<b>Diagrama de Campbell</b>")

ValueError: not enough values to unpack (expected 2, got 0)

In [ ]:
omega_vec   = np.linspace(0.0001, 9000, 100) * (2 * np.pi / 60)  # rad/s
amp_q1_vec  = []
amp_q2_vec  = []
rotor_amortecido.inverse_mass_matrix()
for omega in omega_vec:
    a1, a2  = rotor_amortecido.amplitude_sinal_temporal(omega, t_fim=1, excitacao="desbalanceamento", omega_rpm=4000)
    amp_q1_vec.append(a1)
    amp_q2_vec.append(a2)

In [ ]:
import plotly.graph_objects as go
rotor_amortecido.inverse_mass_matrix()
q1, q2 = rotor_amortecido.resposta_temporal(4000 * 2 * np.pi / 60, excitacao="assincrona", gdl="separado", omega_rpm=4000)
## Convertendo para o domínio físico
u1 = q1 * rotor_amortecido.forma.f(eixo.L / 2)
theta1 = - q1 * rotor_amortecido.forma.g(eixo.L / 2)
u2 = q2 * rotor_amortecido.forma.f(eixo.L / 2)
theta2 = q2 * rotor_amortecido.forma.g(eixo.L / 2)

t = np.arange(0, 1, 1e-4)
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        # x=omega_vec* 60 / (2 * np.pi),
        # y=amp_q1_vec,
        x=t,
        y=u1,
        name=r"$u_{1}$",
    )
)
fig.add_trace(
    go.Scatter(
        # x=omega_vec* 60 / (2 * np.pi),
        # y=amp_q1_vec,
        x=t,
        y=u2,
        name=r"$u_{2}$",
    )
)
fig.show()

np.where()
fig2 = go.Figure()

In [ ]:
fig_dict = rotor_assimetrico.plot_orbita(
    omega_list=[2500, 2640, 3000],
    excitacao="desbalanceamento",
)
for _, fig in fig_dict.items():
    fig.show()